In [ ]:
from openai import OpenAI
import requests
from minsearch import Index
import json

In [29]:
openai_client = OpenAI()

In [30]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [31]:
llm("When does the course start?")

'Could you please provide more details about the specific course you are referring to?'

In [34]:
docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [35]:
documents[11]

{'text': "No, you can only get a certificate if you finish the course with a “live” cohort. We don't award certificates for the self-paced mode. The reason is you need to peer-review capstone(s) after submitting a project. You can only peer-review projects at the time the course is running.",
 'section': 'General course-related questions',
 'question': 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'course': 'data-engineering-zoomcamp'}

## Index Search

In [36]:
# initialize an indexed search engine
index = Index(
    text_fields=["question", "text", "section"], # search lookup fields
    keyword_fields=["course"] # exact filter fields
)

index.fit(documents)

In [37]:
question = 'I just found the course. Can I join now?'

results = index.search(
    question,
    num_results=2
)

results

[{'text': 'Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course.\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers’ Projects by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'section': 'General course-related questions',
  'question': 'The course has already started. Can I still join it?',
  'course': 'machine-learning-zoomcamp'},
 {'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slac

In [38]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5
    )

    return results

In [39]:
search(question)

[{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?',
  'course': 'data-engineering-zoomcamp'},
 {'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
  'section': 'General course-related questions',
  'question': 'Course - When will the course start?',
  'course': 'data-engineerin

In [40]:
instructions = """
    You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
    Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

prompt_template = """
    <QUESTION>
    {question}
    </QUESTION>

    <CONTEXT>
    {context}
    </CONTEXT>
""".strip()

In [41]:
def build_prompt(question, search_results):
    search_json = json.dumps(search_results)
    return prompt_template.format(
        question=question,
        context=search_json
    )

In [42]:
# retries documents relevant to the input question
def rag(question):
    search_results = search(question) # retrieve relevant documents
    user_prompt = build_prompt(question, search_results) # combine search results and user query
    results = llm(user_prompt, instructions=instructions)
    return results

In [43]:
rag(question)

"Yes, you can join the course now. Even if you don’t register, you’re still eligible to submit the homeworks. Just be aware that there will be deadlines for turning in the final projects, so it's best not to leave everything for the last minute."

## Vector Search

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm.auto import tqdm
from minsearch import VectorSearch

In [ ]:
# instantiate transformer model
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

In [ ]:
# get an embedding for each document's question and text
embeddings = []

for d in tqdm(documents):
    text = d['question'] + ' ' + d['text']
    v = embedding_model.encode(text)
    embeddings.append(v)

embeddings = np.array(embeddings)

In [ ]:
# instantiate a vector search engine
vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(embeddings, documents)

In [ ]:
# vector search
def vector_search(question):
    # get embedding for question text
    q = embedding_model.encode(question)

    # perform vector search
    return vindex.search(
        q,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        num_results=5
    )

In [ ]:
# vector-based rag flow
def rag(question):
    search_results = vector_search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt, instructions=instructions)

## Hybrid Search

In [ ]:
def hybrid_search(question):
    r1 = search(question)
    r2 = vector_search(question)
    return r1 + r2